# **Fine-tuning de Mistral 7B sur MedQuAD**
**<u>Objectif:</u>** Adapter le modèle au vocabulaire médical et aux relations symptômes-pathologies.

In [4]:
# Drive pour sauvegardes
# from google.colab import drive
# drive.mount('/content/drive')

In [5]:
# Cloner le dépôt
# !git clone https://github.com/Projet-Capstone-IA/clinical-orientation-ai.git

In [6]:
# Se place dans le répertoire
# %cd clinical-orientation-ai

In [7]:
# Missing packages
%pip install bitsandbytes trl peft


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## **Imports & Configurations**

In [8]:
# Imports standards
import pandas as pd
import numpy as np
import random
import torch
from torch.utils.data import DataLoader
import warnings
import sys
import pickle
import shutil
from tqdm import tqdm
import math

from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback


warnings.filterwarnings('ignore')

In [9]:
# Reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

In [10]:
# Configuration du modèle
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
MAX_LENGTH = 512
BATCH_SIZE = 4  # Ajuster selon mémoire GPU

print(f"Configuration:")
print(f"- Modèle: {MODEL_NAME}")
print(f"- Max length: {MAX_LENGTH}")
print(f"- Batch size: {BATCH_SIZE}")

Configuration:
- Modèle: mistralai/Mistral-7B-v0.1
- Max length: 512
- Batch size: 4


In [11]:
# Vérifier le GPU disponible
print(f"GPU disponible: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Nom du GPU: {torch.cuda.get_device_name(0)}")
    print(f"Mémoire totale: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"Mémoire disponible: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")
else:
    print("Attention: Pas de GPU détecté. L'entraînement sera très lent.")

GPU disponible: True
Nom du GPU: NVIDIA A100-SXM4-80GB
Mémoire totale: 85.29 GB
Mémoire disponible: 0.00 GB


In [12]:
# # Ajouter le chemin pour importer nos modules
import os
import sys

sys.path.append(os.path.abspath(".."))
# sys.path.append('clinical-orientation-ai')

# Importer notre classe Dataset
from utils.dataset import MedQADataset

print(f"Dataset importé avec succès: \n{MedQADataset}")

Dataset importé avec succès: 
<class 'utils.dataset.MedQADataset'>


In [13]:
# Configuration des chemins
from pathlib import Path
PROJET_ROOT = Path.cwd().parent
PROJET_ROOT

PosixPath('/teamspace/studios/this_studio/clinical-orientation-ai')

## **Chargement des données**

In [14]:
# Charger les splits
train_df = pd.read_csv(f"{PROJET_ROOT}/data/train.csv")
val_df = pd.read_csv(f"{PROJET_ROOT}/data/val.csv")
test_df = pd.read_csv(f"{PROJET_ROOT}/data/test.csv")

print(f"Train: {len(train_df)} exemples")
print(f"Validation: {len(val_df)} exemples")
print(f"Test: {len(test_df)} exemples")

# Aperçu
print("\nAperçu des données:")
train_df.head()

Train: 11494 exemples
Validation: 2431 exemples
Test: 2434 exemples

Aperçu des données:


,question,answer,source,focus_area,question_len,answer_len
0,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma,22,1209
1,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma,35,1607
2,Who is at risk for Glaucoma? ?,Anyone can develop glaucoma. Some people are a...,NIHSeniorHealth,Glaucoma,30,493
3,How to prevent Glaucoma ?,"At this time, we do not know how to prevent gl...",NIHSeniorHealth,Glaucoma,25,467
4,What are the symptoms of Glaucoma ?,"At first, open-angle glaucoma has no symptoms....",NIHSeniorHealth,Glaucoma,35,290


## **Chargement des datasets tokenizés**

In [15]:
# Charger les datasets tokenizés sauvegardés
with open('../data/train_dataset.pkl', 'rb') as f:
    train_dataset = pickle.load(f)

with open('../data/val_dataset.pkl', 'rb') as f:
    val_dataset = pickle.load(f)

with open('../data/test_dataset.pkl', 'rb') as f:
    test_dataset = pickle.load(f)

print(f"Train dataset: {len(train_dataset)} exemples")
print(f"Validation dataset: {len(val_dataset)} exemples")
print(f"Test dataset: {len(test_dataset)} exemples")

Train dataset: 11494 exemples
Validation dataset: 2431 exemples
Test dataset: 2434 exemples


In [16]:
# Vérifier un échantillon
sample = train_dataset[0]
print(f"\nClés du dataset: {sample.keys()}")
print(f"input_ids shape: {sample['input_ids'].shape}")
print(f"attention_mask shape: {sample['attention_mask'].shape}")
print(f"labels shape: {sample['labels'].shape}")


Clés du dataset: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids shape: torch.Size([512])
attention_mask shape: torch.Size([512])
labels shape: torch.Size([512])


## **Les DataLoaders**

In [17]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 2874
Validation batches: 608
Test batches: 609


In [18]:
# Tester un batch
batch = next(iter(train_loader))
print(f"\nBatch keys: {batch.keys()}")
print(f"input_ids shape: {batch['input_ids'].shape}")


Batch keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
input_ids shape: torch.Size([4, 512])


## **Configuration du modèle avec QLoRA**

In [19]:
# Connexion
from huggingface_hub import login
# from google.colab import userdata
# login(token=userdata.get('HF_TOKEN'))
login()

In [14]:
# Configuration 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Charger le modèle
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Modèle chargé avec quantification 4-bit")
print(f"Paramètres totals: {model.num_parameters():,}")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Modèle chargé avec quantification 4-bit
Paramètres totals: 7,241,732,096


# **Partie Entraînement - À ignorer si Évaluation (Aller vers la partie `Evaluation`)**

---

## **Configuration LoRA**

In [15]:
# Préparer le modèle pour l'entraînement k-bit
model = prepare_model_for_kbit_training(model)

# Configuration LoRA
lora_config = LoraConfig(
    r=16,  # rang
    lora_alpha=32,  # scaling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # modules à adapter
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Appliquer LoRA
model = get_peft_model(model, lora_config)

print("Configuration LoRA appliquée")
print(f"Paramètres entraînables: {model.num_parameters(only_trainable=True):,}")
print(f"Total paramètres: {model.num_parameters():,}")
print(f"Pourcentage entraînable: {100 * model.num_parameters(only_trainable=True) / model.num_parameters():.2f}%")

Configuration LoRA appliquée
Paramètres entraînables: 13,631,488
Total paramètres: 7,255,363,584
Pourcentage entraînable: 0.19%


## **Configuration des arguments d'entraînement avec early stopping**

In [16]:
# Arguments d'entraînement
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,  # Augmenté
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=250,  # Évaluer plus souvent
    save_steps=250,  # Sauvegarder plus souvent
    learning_rate=2e-4,
    fp16=True,
    save_total_limit=3,  # Garder les 3 meilleurs checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    push_to_hub=False,
    report_to="none",
    logging_dir="./logs",
)

print("Arguments d'entraînement configurés:")
print(f"- Epochs: {training_args.num_train_epochs}")
print(f"- Batch size: {training_args.per_device_train_batch_size}")
print(f"- Learning rate: {training_args.learning_rate}")
print(f"- Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"- Evaluation steps: {training_args.eval_steps}")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Arguments d'entraînement configurés:
- Epochs: 5
- Batch size: 4
- Learning rate: 0.0002
- Gradient accumulation: 4
- Evaluation steps: 250


In [17]:
# Initialiser le trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Trainer initialisé avec early stopping (patience=3)")
print("Les meilleurs poids seront sauvegardés automatiquement")

Trainer initialisé avec early stopping (patience=3)
Les meilleurs poids seront sauvegardés automatiquement


## **Lancement de l'entraînement & Sauvegardes**

In [47]:
# Démarrer l'entraînement
trainer.train()

Step,Training Loss,Validation Loss
250,0.412308,0.416152
500,0.407521,0.407075
750,0.366797,0.401518
1000,0.363304,0.400918
1250,0.364067,0.396039
1500,0.292181,0.407928
1750,0.312934,0.404645
2000,0.315403,0.403145


TrainOutput(global_step=2000, training_loss=0.37022458028793337, metrics={'train_runtime': 8685.4064, 'train_samples_per_second': 6.617, 'train_steps_per_second': 0.414, 'total_flos': 6.99908643398615e+17, 'train_loss': 0.37022458028793337, 'epoch': 2.782185107863605})

In [21]:
# Sauvegarder le modèle et le tokenizer
# model.save_pretrained("models/mistral-medquad-final")
# tokenizer.save_pretrained("./mistral-medquad-final")

# Sauvegarder sur Google Drive
# shutil.make_archive("/content/drive/MyDrive/mistral-medquad-final", 'zip', "./mistral-medquad-final")

# print("Modèle sauvegardé")

---

# **Évaluation du modèle `Mistral-7B`**

In [20]:
from pathlib import Path
# Chemin absolu vers le checkpoint
checkpoint_path = "./results/checkpoint-1250" # Meilleur modèle des 03 runs

In [21]:
# Charger le modèle
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [22]:
print(f"Chemin du checkpoint: {checkpoint_path}")
print(f"Le dossier existe: {os.path.exists(checkpoint_path)}")

Chemin du checkpoint: ./results/checkpoint-1250
Le dossier existe: True


In [23]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Charger les adaptateurs LoRA depuis le checkpoint
model = PeftModel.from_pretrained(base_model, checkpoint_path)

print(f"Modèle de base: {MODEL_NAME}")
print(f"Adaptateurs chargés depuis: {checkpoint_path}")
print(f"Taille: {base_model.num_parameters():,} paramètres")
# print(f"Paramètres entraînables: {model.num_parameters(only_trainable=True):,}")

Modèle de base: mistralai/Mistral-7B-v0.1
Adaptateurs chargés depuis: ./results/checkpoint-1250
Taille: 7,255,363,584 paramètres


In [24]:
# Apercu du modèle
model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32000, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [25]:
print(f"Paramètres totaux: {model.num_parameters():,}")
print(f"Paramètres entraînables: {model.num_parameters(only_trainable=True):,}")
print(f"Pourcentage entraînable: {100 * model.num_parameters(only_trainable=True) / model.num_parameters():.2f}%")

Paramètres totaux: 7,255,363,584
Paramètres entraînables: 0
Pourcentage entraînable: 0.00%


**Paramètres entraînables: 0 car nous sommes en mode évaluation <br />
Quand on charge un modèle avec PeftModel.from_pretrained() sans spécifier is_trainable=True, le modèle est automatiquement en mode inférence**

# **Test d'évaluation sur un échantillon de test**

## Plan d'évaluation du modèle

### **Phase 1 - Évaluation de la reconnaissance des symptômes par pathologie**
Utiliser les paires Q/R de MedQuAD où la question mentionne explicitement une pathologie et la réponse liste ses symptômes.

**Métrique** : F1 score sur les symptômes extraits

### **Phase 2 - Évaluation de l'extraction de symptômes à partir de descriptions patients**
Créer un petit jeu de descriptions patients synthétiques (ou utiliser les questions MedQuAD reformulées)

**Métrique** : Précision/rappel sur les symptômes identifiés

### **Phase 3 - Évaluation de l'association symptômes-pathologies**
Tester si le modèle propose les bonnes pathologies candidates à partir de symptômes

**Métrique** : Recall@3 sur les pathologies

### **Phase 4 - Évaluation des orientations cliniques**
Vérifier si les spécialités proposées correspondent aux pathologies identifiées

**Métrique** : Accuracy spécialité

In [26]:
# Fonction de génération de reponses:
def generate_response(question, model, tokenizer, max_length=256):
    prompt = f"<s>[INST] {question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_length,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("[/INST]")[-1].strip()
    return response

In [27]:
# Prendre un exemple du test set
sample = test_df.iloc[0]
question = sample['question']
true_focus = sample['focus_area']
true_answer = sample['answer']

In [28]:
generated = generate_response(question, model, tokenizer)

In [29]:
print(f"Question: {question}")
print(f"Focus area réel: {true_focus}")
print(f"\nRéponse générée (premiers 300 caractères):")
print(generated[:])
print(f"\nRéponse attendue (premiers 300 caractères):")
print(true_answer[:])

Question: What is (are) Glaucoma ?
Focus area réel: Glaucoma

Réponse générée (premiers 300 caractères):
Glaucoma is a group of diseases that can damage the eye's optic nerve and cause vision loss. The most common type of glaucoma is open-angle glaucoma. It happens when the fluid that flows through the eye builds up, causing increased pressure in the eye. Over time, this increased pressure can damage the optic nerve and cause vision loss. Another type of glaucoma is angle-closure glaucoma. This happens when the angle between the cornea and iris is too narrow. The fluid that drains out of the eye can get blocked, causing a sudden increase in eye pressure. This increase in pressure can damage the optic nerve and cause vision loss. Glaucoma can also be caused by other eye diseases, injuries, or certain medicines. It can also run in families.

Réponse attendue (premiers 300 caractères):
Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blin

In [30]:
# On souhaite que le LLM puisse reconnaitre le contexte clinique:
question_ouverte = "C'est quoi les symptômes du glaucome, je veux uniquement les symptômes, réponds moi en franćais."
reponse_generee = generate_response(question_ouverte, model, tokenizer)

print(f"Question: {question_ouverte}")
print(generated[:])

Question: C'est quoi les symptômes du glaucome, je veux uniquement les symptômes, réponds moi en franćais.
Glaucoma is a group of diseases that can damage the eye's optic nerve and cause vision loss. The most common type of glaucoma is open-angle glaucoma. It happens when the fluid that flows through the eye builds up, causing increased pressure in the eye. Over time, this increased pressure can damage the optic nerve and cause vision loss. Another type of glaucoma is angle-closure glaucoma. This happens when the angle between the cornea and iris is too narrow. The fluid that drains out of the eye can get blocked, causing a sudden increase in eye pressure. This increase in pressure can damage the optic nerve and cause vision loss. Glaucoma can also be caused by other eye diseases, injuries, or certain medicines. It can also run in families.


In [31]:
# Tester une question spécifique aux symptômes en anglais
question_symptoms = "What are the symptoms of glaucoma? Please list only the symptoms."
response = generate_response(question_symptoms, model, tokenizer)

print(f"Question: {question_symptoms}")
print(f"\nRéponse générée:")
print(response)

Question: What are the symptoms of glaucoma? Please list only the symptoms.

Réponse générée:
What are the signs and symptoms of glaucoma? The Human Phenotype Ontology provides the following list of signs and symptoms for glaucoma. If the information is available, the table below includes how often the symptom is seen in people with this condition. You can use the MedlinePlus Medical Dictionary to look up the definitions for these medical terms. Signs and Symptoms Approximate number of patients (when available) Abnormality of the eye - Autosomal dominant inheritance - Glaucoma - The Human Phenotype Ontology (HPO) has collected information on how often a sign or symptom occurs in a condition. Much of this information comes from Orphanet, a European rare disease database. The frequency of a sign or symptom is usually listed as a rough estimate of the percentage of patients who have that feature. The frequency may also be listed as a fraction. The first number of the fraction is how many 

In [32]:
# Extraire les symptômes de la réponse (approche simple)
symptoms_keywords = ["pain", "vision", "blur", "headache", "nausea", "redness", "halos"]
found_symptoms = [kw for kw in symptoms_keywords if kw in response.lower()]
print(f"\nSymptômes détectés dans la réponse: {found_symptoms}")


Symptômes détectés dans la réponse: []


# **Évaluation de la `Perplexity` sur le test set**
Mesure la capacité du modèle à **reproduire la distribution linguistique du corpus**. Utile pour vérifier la convergence et la stabilité du modèle lors de l’entraînement, mais elle reste un indicateur secondaire car elle ne reflète pas directement la performance clinique ou la pertinence des diagnostics (diagnostic non-fourni)

In [33]:
# Fonction pour évaluer la perplexité:
def compute_perplexity(model, dataloader):
    model.eval()
    total_loss = 0
    total_tokens = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Calcul perplexité"):
            input_ids = batch['input_ids'].to(model.device)
            attention_mask = batch['attention_mask'].to(model.device)
            labels = batch['labels'].to(model.device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            
            # Nombre de tokens non masqués dans ce batch
            batch_tokens = (labels != -100).sum().item()
            
            total_loss += loss.item() * batch_tokens
            total_tokens += batch_tokens
    
    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    
    return perplexity, avg_loss

In [34]:
# Calculer la perplexité sur le test set
test_perplexity, test_loss = compute_perplexity(model, test_loader)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Perplexity: {test_perplexity:.2f}")

Calcul perplexité:   0%|          | 0/609 [00:00<?, ?it/s]

Calcul perplexité: 100%|██████████| 609/609 [02:05<00:00,  4.86it/s]

Test Loss: 0.3777
Test Perplexity: 1.46


**Interprétation :**<br />
- **Perplexity = 1.46** signifie que le modèle est très confiant dans ses prédictions
- Plus la perplexité est basse, mieux c'est
- Une valeur de 1.0 serait parfaite (le modèle est certain à 100%)



**Échelle de référence :** <br />
- 1.0 : Parfait (modèle certain)
- 1.5 : Très bon (c'est notre cas)
- 2.0 : Bon
- 5.0 : Médiocre
- 10+ : Mauvais

## Rappel de l'objectif

Développer un agent conversationnel qui :
- Engage le patient dans un dialogue naturel
- Recueille les symptômes décrits par le patient
- Génère une carte d'orientation clinique exploratoire (arbre décisionnel)
- Sans diagnostic, sans conclusion, sans niveau d'urgence

## Rôle du modèle fine-tuné
- Comprendre le vocabulaire médical et les relations symptômes-pathologies
- Générer des réponses qui explorent les possibilités plutôt que de conclure

# **Évaluation de `Exact Match (EM)` sur les questions simples**

In [35]:
# Fonction qui calcule exact match sur des questions simples:
def calculate_exact_match(model, tokenizer, test_df, num_samples=100):
    """
    Calcule l'Exact Match entre les réponses générées et les réponses attendues
    """
    model.eval()
    correct = 0
    total = 0
    
    # Échantillonner pour aller plus vite
    sample_df = test_df.sample(n=min(num_samples, len(test_df)), random_state=42)
    
    for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
        question = row['question']
        true_answer = row['answer']
        
        # Générer la réponse
        prompt = f"<s>[INST] {question} [/INST]"
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.1,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
        generated = generated.split("[/INST]")[-1].strip()
        
        # Normalisation simple
        true_norm = true_answer.strip().lower()
        gen_norm = generated.strip().lower()
        
        # Exact match sur les 100 premiers caractères (pour éviter les variations mineures)
        if true_norm[:100] == gen_norm[:100]:
            correct += 1
        total += 1
    
    em_score = correct / total
    return em_score

In [36]:
# Calculer l'Exact Match
em_score = calculate_exact_match(model, tokenizer, test_df, num_samples=100)
print(f"Exact Match (échantillon 100): {em_score:.2%}")

  0%|          | 0/100 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


100%|██████████| 100/100 [17:58<00:00, 10.78s/it]

Exact Match (échantillon 100): 23.00%
